# Cayley/RoBERTa Colab Runtime Setup

This notebook runs on a Colab GPU kernel while loading your project files from Drive, GitHub, or an uploaded zip.

## 1. Runtime check

In Colab, use `Runtime > Change runtime type > GPU` before running the rest of the notebook.

In [ ]:
import os
import platform
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())
!nvidia-smi || true

## 2. Choose project source

Use `drive` if your project folder is in Google Drive. Use `github` if the repo is pushed. Use `upload` for a zipped copy of the project.

In [ ]:
PROJECT_SOURCE = 'drive'  # 'drive', 'github', or 'upload'

# Drive settings
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/cayley'

# GitHub settings. Leave GITHUB_TOKEN empty for public repos.
GITHUB_REPO = 'https://github.com/YOUR_USER/YOUR_REPO.git'
GITHUB_BRANCH = 'main'
GITHUB_TOKEN = ''

RUNTIME_PROJECT_DIR = '/content/cayley'
REQUIREMENTS_FILE = 'requirements-colab.txt'

## 3. Make project files available to Colab

In [ ]:
from pathlib import Path
import shutil
import subprocess

def run(cmd, cwd=None):
    print('+', cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

if PROJECT_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = Path(DRIVE_PROJECT_DIR)
    if not project_dir.exists():
        raise FileNotFoundError(f'Drive project folder not found: {project_dir}')

elif PROJECT_SOURCE == 'github':
    repo_url = GITHUB_REPO
    if GITHUB_TOKEN:
        repo_url = repo_url.replace('https://', f'https://{GITHUB_TOKEN}@')
    project_dir = Path(RUNTIME_PROJECT_DIR)
    if project_dir.exists():
        shutil.rmtree(project_dir)
    run(f'git clone --branch {GITHUB_BRANCH} --depth 1 {repo_url} {project_dir}')

elif PROJECT_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise ValueError('Upload a .zip file containing the project.')
    project_dir = Path(RUNTIME_PROJECT_DIR)
    if project_dir.exists():
        shutil.rmtree(project_dir)
    project_dir.mkdir(parents=True)
    run(f'unzip -q {zip_names[0]} -d {project_dir}')
    children = [p for p in project_dir.iterdir() if p.is_dir()]
    if len(children) == 1 and not (project_dir / REQUIREMENTS_FILE).exists():
        project_dir = children[0]

else:
    raise ValueError(f'Unknown PROJECT_SOURCE: {PROJECT_SOURCE}')

print('Project dir:', project_dir)
print('Top-level files:', sorted(p.name for p in project_dir.iterdir())[:30])

## 4. Install dependencies and project

In [ ]:
import sys
from pathlib import Path

req_path = Path(project_dir) / REQUIREMENTS_FILE
if req_path.exists():
    run(f'{sys.executable} -m pip install -q -r {req_path}')
else:
    run(f'{sys.executable} -m pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121')
    run(f'{sys.executable} -m pip install -q transformers datasets evaluate accelerate scikit-learn networkx pandas tqdm matplotlib seaborn einops wandb')

if (Path(project_dir) / 'pyproject.toml').exists() or (Path(project_dir) / 'setup.py').exists():
    run(f'{sys.executable} -m pip install -q -e {project_dir}')

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print('sys.path[0]:', sys.path[0])

## 5. Verify PyTorch, Transformers, and RoBERTa

In [ ]:
import torch
import transformers
from transformers import AutoConfig, AutoModel, AutoTokenizer

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
print('transformers:', transformers.__version__)

model_name = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, config=config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

inputs = tokenizer('cayley graph transformer pattern benchmark smoke test', return_tensors='pt').to(device)
with torch.no_grad():
    outputs = model(**inputs)
print('last_hidden_state:', tuple(outputs.last_hidden_state.shape))

## 6. Run your benchmark

Set `BENCHMARK_COMMAND` to the script you want to run from the project root.

In [ ]:
BENCHMARK_COMMAND = ''  # example: 'python scripts/benchmark_roberta_cayley.py --model roberta-base'

if BENCHMARK_COMMAND:
    run(BENCHMARK_COMMAND, cwd=project_dir)
else:
    print('Set BENCHMARK_COMMAND when your benchmark entrypoint is ready.')